### apple, bean, pepper, rice 관찰데이터에 지점ID 매핑

In [ ]:
# pip install pyproj
import os, numpy as np, pandas as pd
from pyproj import Transformer

FILES = {
    "APPLE":  "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE.csv",
    "BEAN":   "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN.csv",
    "PEPPER": "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER.csv",
    "RICE":   "/Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE.csv",
}

CELL_M = 30
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

for crop, path in FILES.items():
    if not os.path.exists(path):
        print(f"[건너뜀] {crop}: {path} 없음"); 
        continue

    # 1) 파일마다 DF 새로 로드
    df = pd.read_csv(path, encoding="utf-8-sig")

    # 2) 좌표 숫자화 + 유효 마스크
    df["좌표-경도"] = pd.to_numeric(df.get("좌표-경도"), errors="coerce")
    df["좌표-위도"] = pd.to_numeric(df.get("좌표-위도"), errors="coerce")
    valid = df["좌표-경도"].notna() & df["좌표-위도"].notna()

    # 3) 좌표 변환(유효 행만)
    x = np.full(len(df), np.nan)
    y = np.full(len(df), np.nan)
    if valid.any():
        xv, yv = tf.transform(df.loc[valid, "좌표-경도"].values,
                              df.loc[valid, "좌표-위도"].values)
        x[valid], y[valid] = xv, yv

    # 4) 30m 격자 index → 지점ID 생성
    ix = pd.Series(np.floor_divide(x, CELL_M)).astype("Int64")
    iy = pd.Series(np.floor_divide(y, CELL_M)).astype("Int64")

    df["지점ID"] = pd.NA
    df.loc[valid, "지점ID"] = ix[valid].astype(str).str.cat(iy[valid].astype(str), sep="_")

    # (선택) 지점ID를 맨 앞으로
    cols = ["지점ID"] + [c for c in df.columns if c != "지점ID"]
    df = df[cols]

    # 5) 파일별로 각각 저장
    base, ext = os.path.splitext(path)
    out_path = f"{base}_with_siteid{ext}"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {crop}: {out_path} | rows={len(df):,}, 지점ID 생성={df['지점ID'].notna().sum():,}")


[저장] APPLE: /Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_APPLE_with_siteid.csv | rows=12,217, 지점ID 생성=12,217
[저장] BEAN: /Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_BEAN_with_siteid.csv | rows=5,231, 지점ID 생성=5,231
[저장] PEPPER: /Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_PEPPER_with_siteid.csv | rows=24,091, 지점ID 생성=24,091
[저장] RICE: /Users/doyoung-gil/Desktop/연구실/AIM_PEX_DT03_샘플데이터/RDA_OBSERVATION_RICE_with_siteid.csv | rows=80,733, 지점ID 생성=80,709


### 조사일자 datetime 형식으로 변환(기상데이터와 join를 위해)

In [ ]:
import pandas as pd
import os

# 지점ID까지 붙여둔 파일들(_with_siteid.csv)
FILES = {
    "APPLE":  "/Users/doyoung-gil/Desktop/연구실/데이터/RDA_OBSERVATION_APPLE_with_siteid.csv",
    "BEAN":   "/Users/doyoung-gil/Desktop/연구실/데이터/RDA_OBSERVATION_BEAN_with_siteid.csv",
    "PEPPER": "/Users/doyoung-gil/Desktop/연구실/데이터/RDA_OBSERVATION_PEPPER_with_siteid.csv",
    "RICE":   "/Users/doyoung-gil/Desktop/연구실/데이터/RDA_OBSERVATION_RICE_with_siteid.csv",
}

OVERWRITE = False  # True로 바꾸면 _with_siteid.csv를 직접 덮어씀

for crop, path in FILES.items():
    if not os.path.exists(path):
        print(f"[건너뜀] {crop}: 파일 없음 → {path}")
        continue

    df = pd.read_csv(path, encoding="utf-8-sig")

    if "조사일자(YYYYMMDD)" not in df.columns:
        print(f"[경고] {crop}: '조사일자(YYYYMMDD)' 컬럼 없음")
        continue

    # 문자열 → datetime (기존 컬럼 덮어쓰기)
    before_na = df["조사일자(YYYYMMDD)"].isna().sum()
    df["조사일자(YYYYMMDD)"] = pd.to_datetime(
        df["조사일자(YYYYMMDD)"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )
    after_na = df["조사일자(YYYYMMDD)"].isna().sum()

    # (선택) 정렬: 지점ID → 조사일자
    if "지점ID" in df.columns:
        df = df.sort_values(["지점ID", "조사일자(YYYYMMDD)"])

    # 저장 경로
    if OVERWRITE:
        out_path = path
    else:
        base, ext = os.path.splitext(path)
        out_path = f"{base}_dt{ext}"

    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {crop}: {out_path} | NaN(before→after) {before_na}→{after_na}, rows={len(df):,}")


[저장] APPLE: /Users/doyoung-gil/Desktop/연구실/기상데이터/RDA_OBSERVATION_APPLE_with_siteid_dt.csv | NaN(before→after) 0→0, rows=12,217
[저장] BEAN: /Users/doyoung-gil/Desktop/연구실/기상데이터/RDA_OBSERVATION_BEAN_with_siteid_dt.csv | NaN(before→after) 0→0, rows=5,231
[저장] PEPPER: /Users/doyoung-gil/Desktop/연구실/기상데이터/RDA_OBSERVATION_PEPPER_with_siteid_dt.csv | NaN(before→after) 0→0, rows=24,091
[저장] RICE: /Users/doyoung-gil/Desktop/연구실/기상데이터/RDA_OBSERVATION_RICE_with_siteid_dt.csv | NaN(before→after) 0→0, rows=80,733


### 작물별 병해충 데이터 병합, 년도별 나누기

In [11]:
import os
import pandas as pd

# 1) 입력 파일: 지점ID까지 붙이고 날짜도 변환해둔 파일들
FILES = {
    "APPLE":  "/Users/doyoung-gil/Desktop/연구실/기상데이터/관찰데이터+지점ID+datetime/RDA_OBSERVATION_APPLE_with_siteid_dt.csv",
    "BEAN":   "/Users/doyoung-gil/Desktop/연구실/기상데이터/관찰데이터+지점ID+datetime/RDA_OBSERVATION_BEAN_with_siteid_dt.csv",
    "PEPPER": "/Users/doyoung-gil/Desktop/연구실/기상데이터/관찰데이터+지점ID+datetime/RDA_OBSERVATION_PEPPER_with_siteid_dt.csv",
    "RICE":   "/Users/doyoung-gil/Desktop/연구실/기상데이터/관찰데이터+지점ID+datetime/RDA_OBSERVATION_RICE_with_siteid_dt.csv",
}

OUT_DIR = "/Users/doyoung-gil/Desktop/연구실/ANTH_by_year"
os.makedirs(OUT_DIR, exist_ok=True)

# 탄저병 컬럼 키워드(필요시 추가)
KEYS = ["탄저", "탄저병"]

# ✅ 메타 컬럼 순서(요청한 순서)
META = [
    "지점ID", "조사년도", "지역-시도", "지역-시군구", "지역-읍면동",
    "상세주소", "좌표-경도", "좌표-위도", "조사회차",
    "조사일자(YYYYMMDD)",  # ← 여기 뒤에 작물명 배치
    "작물명", "품종명", "조사구분"
]

def load_one(path, crop_label):
    df = pd.read_csv(path, encoding="utf-8-sig")
    # 작물명 없으면 파일명 라벨로 채움
    if "작물명" not in df.columns:
        df["작물명"] = crop_label

    # 날짜 안전 변환(이미 datetime이어도 OK)
    if "조사일자(YYYYMMDD)" in df.columns:
        df["조사일자(YYYYMMDD)"] = pd.to_datetime(
            df["조사일자(YYYYMMDD)"], errors="coerce"
        )

    # 탄저 관련 컬럼
    anth_cols = [c for c in df.columns if any(k in c for k in KEYS)]

    # 요청한 메타 순서 + 탄저 컬럼만 유지(존재하는 것만)
    keep = [c for c in META if c in df.columns] + anth_cols
    if not anth_cols:
        print(f"[경고] {crop_label}: 탄저 관련 컬럼을 찾지 못함 → 건너뜀 ({os.path.basename(path)})")
        return None

    out = df[keep].copy()

    # 탄저값 전부 결측 행 제거(원하면 주석 처리)
    out = out.dropna(subset=anth_cols, how="all")
    return out

# 2) 네 파일 합치기
parts = []
for crop, p in FILES.items():
    if not os.path.exists(p):
        print(f"[건너뜀] {crop}: 파일 없음 → {p}")
        continue
    part = load_one(p, crop)
    if part is None or part.empty:  
        continue
    parts.append(part)

if not parts:
    raise SystemExit("합칠 데이터가 없습니다. 경로/파일을 확인하세요.")

all_df = pd.concat(parts, ignore_index=True)

# 정렬(지점ID → 날짜)
if {"지점ID","조사일자(YYYYMMDD)"}.issubset(all_df.columns):
    all_df = all_df.sort_values(["지점ID","조사일자(YYYYMMDD)"])

# 3) 연도별 저장
if "조사일자(YYYYMMDD)" not in all_df.columns:
    raise ValueError("'조사일자(YYYYMMDD)' 컬럼이 없습니다.")

all_df["year"] = all_df["조사일자(YYYYMMDD)"].dt.year

for y, sub in all_df.groupby("year"):
    if pd.isna(y):
        bad_path = os.path.join(OUT_DIR, "ANTH_year_unknown.csv")
        sub.drop(columns=["year"]).to_csv(bad_path, index=False, encoding="utf-8-sig")
        print(f"[저장] {bad_path} (rows={len(sub):,})")
        continue

    save_path = os.path.join(OUT_DIR, f"ANTH_{int(y)}.csv")
    sub.drop(columns=["year"]).to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_path} (rows={len(sub):,})")

# 합본
all_save = os.path.join(OUT_DIR, "ANTH_all_years.csv")
all_df.drop(columns=["year"]).to_csv(all_save, index=False, encoding="utf-8-sig")
print(f"[저장] {all_save} (rows={len(all_df):,})")


[경고] RICE: 탄저 관련 컬럼을 찾지 못함 → 건너뜀 (RDA_OBSERVATION_RICE_with_siteid_dt.csv)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2011.csv (rows=167)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2012.csv (rows=231)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2013.csv (rows=431)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2014.csv (rows=2,253)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2015.csv (rows=2,431)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2016.csv (rows=3,389)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2017.csv (rows=3,661)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2018.csv (rows=3,628)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2019.csv (rows=3,805)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2020.csv (rows=3,590)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2021.csv (rows=3,796)
[저장] /Users/doyoung-gil/Desktop/연구실/ANTH_by_year/ANTH_2022.csv (rows=3,320)
[저장] /Users/doyoung

### 년도별 병해충 데이터와 기상데이터 join

In [12]:
import os, pandas as pd, numpy as np
from scipy.spatial import cKDTree

PEST_DIR = "/Users/doyoung-gil/Desktop/연구실/ANTH_by_year"
WEATH_DIR = "/Users/doyoung-gil/Desktop/연구실/기상데이터/ASOS+AWS"
OUT_DIR = "/Users/doyoung-gil/Desktop/연구실/joined_by_year"
os.makedirs(OUT_DIR, exist_ok=True)

def load_weather(year):
    fp = os.path.join(WEATH_DIR, f"ASOS_AWS_{year}.csv")
    w = pd.read_csv(fp, encoding="utf-8-sig")
    # datetime 정리
    w["일시"] = pd.to_datetime(w["일시"], errors="coerce").dt.normalize()
    # 숫자화 + 중복정리
    for c in ["지점","위도","경도"]:
        if c in w.columns: w[c] = pd.to_numeric(w[c], errors="coerce")
    w = w.drop_duplicates(subset=["지점","일시"]).sort_values(["지점","일시"])
    return w

for fn in sorted(os.listdir(PEST_DIR)):
    if not (fn.startswith("ANTH_") and fn.endswith(".csv")): 
        continue
    year = fn.split("_")[1].split(".")[0]
    try:
        y = int(year)
    except:
        continue

    # 1) 병해(그 해) 로드
    pest = pd.read_csv(os.path.join(PEST_DIR, fn), encoding="utf-8-sig")
    if "조사일자(YYYYMMDD)" not in pest.columns or "지점ID" not in pest.columns:
        print(f"[건너뜀] {fn}: 필수 컬럼 없음"); continue
    pest["조사일자(YYYYMMDD)"] = pd.to_datetime(pest["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()

    # 2) 기상(그 해) 로드
    try:
        w = load_weather(y)
    except FileNotFoundError:
        print(f"[건너뜀] {y}: 기상 파일 없음"); 
        continue

    # 3) 지점ID별 좌표 → 최근접 관측소 매핑
    if not {"좌표-위도","좌표-경도"}.issubset(pest.columns):
        print(f"[경고] {fn}: 좌표 컬럼 없어 매핑 불가"); 
        continue
    site_xy = (pest[["지점ID","좌표-위도","좌표-경도"]]
               .dropna().drop_duplicates()
               .rename(columns={"좌표-위도":"p_lat","좌표-경도":"p_lon"}))

    stn = w[["지점","위도","경도"]].dropna().drop_duplicates()
    if stn.empty or site_xy.empty:
        print(f"[건너뜀] {y}: 관측소/지점 좌표 비어있음"); 
        continue

    tree = cKDTree(stn[["위도","경도"]].to_numpy())
    dist, idx = tree.query(site_xy[["p_lat","p_lon"]].to_numpy(), k=1)
    site_xy["최근접_지점"] = stn.iloc[idx]["지점"].to_numpy()
    site_xy["관측소거리_km"] = dist * 111.0

    # 4) 관측소의 모든 날짜를 지점ID로 복제
    base = site_xy.merge(w, left_on="최근접_지점", right_on="지점", how="left")

    # 5) 병해(조사일만 값) LEFT JOIN
    pest_small = pest.drop_duplicates(subset=["지점ID","조사일자(YYYYMMDD)"])
    joined = base.merge(
        pest_small,
        left_on=["지점ID","일시"],
        right_on=["지점ID","조사일자(YYYYMMDD)"],
        how="left",
        suffixes=("","_p")
    )

    # 6) 보기 좋게 배치/정렬
    head = ["지점ID","작물명","품종명","조사구분","p_lat","p_lon",
            "최근접_지점","지점명","위도","경도","관측소거리_km","일시"]
    front = [c for c in head if c in joined.columns]
    rest  = [c for c in joined.columns if c not in front]
    joined = joined[front + rest].sort_values(["지점ID","일시"])

    out_path = os.path.join(OUT_DIR, f"joined_{y}.csv")
    joined.to_csv(out_path, index=False, encoding="utf-8-sig")

    miss = joined.filter(like="탄저").isna().all(axis=1).mean() if any("탄저" in c for c in joined.columns) else np.nan
    print(f"[저장] {out_path}  rows={len(joined):,}  병해없음비율≈{miss if not np.isnan(miss) else 'N/A'}")


[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2011.csv  rows=20,419  병해없음비율≈0.9918213428669377
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2012.csv  rows=20,678  병해없음비율≈0.9889737885675597
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2013.csv  rows=30,286  병해없음비율≈0.9946179753021198
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2014.csv  rows=102,433  병해없음비율≈0.9806605293216053
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2015.csv  rows=112,065  병해없음비율≈0.9798599027350199
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2016.csv  rows=152,775  병해없음비율≈0.9787007036491573
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2017.csv  rows=166,173  병해없음비율≈0.9793047005229489
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2018.csv  rows=171,274  병해없음비율≈0.9799035463643051
[저장] /Users/doyoung-gil/Desktop/연구실/joined_by_year/joined_2019.csv  rows=174,441  병해없음비율≈0.979293858668547
[저장] /Users/doyoung-gil/Desktop/

### 작물별 각 병해충 데이터 추출

In [15]:
import os, re, glob
import pandas as pd

INPUT_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/관찰데이터+지점ID+datetime"
OUT_DIR   = "/Users/doyoung-gil/Desktop/연구실/데이터/pest_by_crop"
os.makedirs(OUT_DIR, exist_ok=True)

PEST_NAME = "탄저병"
PEST_KEYS = ["탄저", "탄저병"]

# ▶ 표준(Core) 스키마: 모델/조인에 공통으로 쓸 컬럼
CORE = [
    "지점ID", "조사년도", "지역-시도", "지역-시군구", "지역-읍면동",
    "상세주소", "좌표-경도", "좌표-위도", "조사회차",
    "조사일자(YYYYMMDD)",  # ← 뒤에 작물명
    "작물명", "품종명", "조사구분"  # ← BEAN에만 있어도 모두에 강제로 만듦
]

# ▶ Extras 보존 여부/방식
KEEP_EXTRAS = False        # False면 extras 완전 제거
PREFIX_EXTRAS = True       # True면 'CROP__컬럼명' 접두어로 이름 충돌 방지

def detect_crop_from_name(fname: str) -> str:
    m = re.search(r"RDA_OBSERVATION_([A-Za-z]+)_with_siteid_dt\.csv", fname)
    return m.group(1).upper() if m else "UNKNOWN"

def load_one(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    # 작물명 보정
    if "작물명" not in df.columns:
        df["작물명"] = detect_crop_from_name(os.path.basename(path))

    # 날짜를 datetime으로(열 이름 유지)
    if "조사일자(YYYYMMDD)" in df.columns:
        df["조사일자(YYYYMMDD)"] = pd.to_datetime(df["조사일자(YYYYMMDD)"], errors="coerce")

    # 병해 컬럼만 선별
    pest_cols = [c for c in df.columns if any(k.casefold() in c.casefold() for k in PEST_KEYS)]
    if not pest_cols:
        print(f"[건너뜀] {os.path.basename(path)}: '{PEST_NAME}' 관련 컬럼 없음")
        return pd.DataFrame()

    # 모든 데이터셋에 '조사구분' 강제 생성(없으면 NaN)
    if "조사구분" not in df.columns:
        df["조사구분"] = pd.NA

    # Core + 병해
    core_present = [c for c in CORE if c in df.columns]   # 실제 존재하는 Core만
    out = df[core_present + pest_cols].copy()

    # 병해값이 전부 결측인 행 제거(원치 않으면 주석 처리)
    out = out.dropna(subset=pest_cols, how="all")

    # Extras (옵션): Core/pest에 없는 나머지를 추가
    if KEEP_EXTRAS:
        used = set(core_present + pest_cols)
        extras = [c for c in df.columns if c not in used]
        if extras:
            if PREFIX_EXTRAS:
                crop = out["작물명"].iloc[0] if not out.empty else detect_crop_from_name(os.path.basename(path))
                extra_ren = {c: f"{crop}__{c}" for c in extras}
                out = pd.concat([out, df.loc[out.index, extras].rename(columns=extra_ren)], axis=1)
            else:
                out = pd.concat([out, df.loc[out.index, extras]], axis=1)

    # 정렬
    if {"지점ID","조사일자(YYYYMMDD)"} <= set(out.columns):
        out = out.sort_values(["지점ID","조사일자(YYYYMMDD)"])

    return out

# === 메인: 작물별로 저장 ===
files = sorted(glob.glob(os.path.join(INPUT_DIR, "RDA_OBSERVATION_*_with_siteid_dt.csv")))
by_crop = {}

for fp in files:
    df = load_one(fp)
    if df.empty:
        continue
    crop = df["작물명"].iloc[0]
    by_crop.setdefault(crop, []).append(df)

if not by_crop:
    raise SystemExit("추출할 데이터가 없습니다. 경로/파일/키워드를 확인하세요.")

for crop, parts in by_crop.items():
    all_df = pd.concat(parts, ignore_index=True)

    # Core 먼저, 그 다음 병해 컬럼, 그 다음 Extras
    core_present = [c for c in CORE if c in all_df.columns]
    pest_cols = [c for c in all_df.columns
                 if c not in core_present and any(k.casefold() in c.casefold() for k in PEST_KEYS)]
    extras = [c for c in all_df.columns if c not in core_present + pest_cols]

    all_df = all_df[core_present + pest_cols + extras]

    # 최종 정렬
    if {"지점ID","조사일자(YYYYMMDD)"} <= set(all_df.columns):
        all_df = all_df.sort_values(["지점ID","조사일자(YYYYMMDD)"])

    save_all = os.path.join(OUT_DIR, f"{PEST_NAME}_{crop}_all_years.csv")
    all_df.to_csv(save_all, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_all} (rows={len(all_df):,}, cols={len(all_df.columns)})")


[건너뜀] RDA_OBSERVATION_RICE_with_siteid_dt.csv: '탄저병' 관련 컬럼 없음
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/pest_by_crop/탄저병_APPLE_all_years.csv (rows=9,528, cols=15)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/pest_by_crop/탄저병_BEAN_all_years.csv (rows=4,251, cols=15)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/pest_by_crop/탄저병_PEPPER_all_years.csv (rows=23,649, cols=14)


### 연도별 기상데이터 -> 전연도 통합(csv, parquet)

In [16]:
import os, glob
import pandas as pd

WEATHER_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"
OUT_CSV  = os.path.join(WEATHER_DIR, "ASOS_AWS_ALL.csv")
OUT_PARQ = os.path.join(WEATHER_DIR, "ASOS_AWS_ALL.parquet")

# 1) 연도별 파일 모으기
files = sorted(glob.glob(os.path.join(WEATHER_DIR, "ASOS_AWS_*.csv")))
if not files:
    raise SystemExit(f"연도별 파일을 찾지 못했습니다: {WEATHER_DIR}/ASOS_AWS_*.csv")

parts = []
for fp in files:
    df = pd.read_csv(fp, encoding="utf-8-sig")

    # 날짜 파싱(자정 정규화)
    if "일시" in df.columns:
        df["일시"] = pd.to_datetime(df["일시"], errors="coerce").dt.normalize()

    # 수치형 보정
    for c in ("지점", "위도", "경도"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    parts.append(df)

# 2) 통합 + 중복 제거 + 정렬
all_df = pd.concat(parts, ignore_index=True)

if {"지점", "일시"}.issubset(all_df.columns):
    all_df = (all_df
              .drop_duplicates(subset=["지점", "일시"])
              .sort_values(["지점", "일시"])
              .reset_index(drop=True))

# 3) 저장 (CSV 필수, Parquet 선택)
all_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"[저장] {OUT_CSV}  rows={len(all_df):,}, cols={len(all_df.columns)}")

try:
    all_df.to_parquet(OUT_PARQ, index=False)  # pyarrow 필요
    print(f"[저장] {OUT_PARQ}")
except Exception as e:
    print(f"[참고] Parquet 저장 건너뜀: {e}")

# 4) 관측소 메타(최근접 매핑용)도 따로 보관
meta_cols = [c for c in ("지점","지점명","위도","경도") if c in all_df.columns]
if meta_cols:
    stn = all_df[meta_cols].dropna().drop_duplicates().sort_values("지점")
    meta_path = os.path.join(WEATHER_DIR, "ASOS_AWS_station_meta.csv")
    stn.to_csv(meta_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {meta_path}  stations={len(stn)}")


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_ALL.csv  rows=3,096,203, cols=11
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_ALL.parquet
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_station_meta.csv  stations=668


### 샘플 : 한 작물에서 한 연도 기상데이터 + 병해충데이터 

In [ ]:
# sample_join_one.py
import os, re, glob
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ========== 설정 ==========
PEST_NAME = "탄저병"
PEST_KEYS = ["탄저", "탄저병"]
CROP = "APPLE"        # "APPLE" / "BEAN" / "PEPPER" / "RICE"
YEAR = 2018           # 예: 2021로 고정 가능. None이면 사용 가능한 최신 연도 자동 선택.

WEATHER_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"   # ASOS_AWS_YYYY.csv 위치
PEST_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/pest_by_crop"         # 탄저병_APPLE_all_years.csv 등
OUT_DIR     = "/Users/doyoung-gil/Desktop/연구실/데이터/joined_samples"
os.makedirs(OUT_DIR, exist_ok=True)
# ==========================

def load_weather_year(y: int) -> pd.DataFrame:
    fp = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}.csv")
    if not os.path.exists(fp):
        raise FileNotFoundError(f"기상 연도 파일 없음: {fp}")
    w = pd.read_csv(fp, encoding="utf-8-sig")
    # 타입 정리
    w["일시"] = pd.to_datetime(w["일시"], errors="coerce").dt.normalize()
    for c in ["지점","위도","경도"]:
        if c in w.columns:
            w[c] = pd.to_numeric(w[c], errors="coerce")
    # 중복 제거
    if {"지점","일시"}.issubset(w.columns):
        w = w.drop_duplicates(subset=["지점","일시"]).sort_values(["지점","일시"])
    return w

def pest_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if any(k.casefold() in c.casefold() for k in PEST_KEYS)]

# 1) 병해 (작물별 all_years) 로드
p_path = os.path.join(PEST_DIR, f"{PEST_NAME}_{CROP}_all_years.csv")
if not os.path.exists(p_path):
    raise SystemExit(f"병해 파일 없음: {p_path}")

p_all = pd.read_csv(p_path, encoding="utf-8-sig")
if "조사일자(YYYYMMDD)" not in p_all.columns:
    raise SystemExit("병해 파일에 '조사일자(YYYYMMDD)' 컬럼이 필요합니다.")

p_all["일시"] = pd.to_datetime(p_all["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()

required = {"지점ID","좌표-위도","좌표-경도","일시"}
missing = required - set(p_all.columns)
if missing:
    raise SystemExit(f"병해 파일에 필요한 컬럼 없음: {missing}")

# 2) 사용할 연도 결정
years_in_pest = sorted(p_all["일시"].dt.year.dropna().unique().astype(int))
if not years_in_pest:
    raise SystemExit("병해 데이터에서 사용 가능한 연도를 찾지 못했습니다.")
y = YEAR if YEAR is not None else years_in_pest[-1]

# 3) 해당 연도 데이터 준비
w = load_weather_year(y)
days = pd.DataFrame({"일시": pd.date_range(f"{y}-01-01", f"{y}-12-31", freq="D")})

# 4) 최근접 관측소 매핑 (지점ID → 지점)
site_xy = p_all[["지점ID","좌표-위도","좌표-경도"]].dropna().drop_duplicates()
stn = w[["지점","위도","경도"]].dropna().drop_duplicates()
if stn.empty:
    raise SystemExit(f"{y}년 기상에 관측소 좌표 정보가 없습니다.")
tree = cKDTree(stn[["위도","경도"]].to_numpy())
dist, idx = tree.query(site_xy[["좌표-위도","좌표-경도"]].to_numpy(), k=1)
site_map = site_xy.assign(지점=stn.iloc[idx]["지점"].to_numpy())[["지점ID","지점"]]

# 5) 기준표(왼쪽): 지점ID × 연중 모든 날짜
base = site_map.assign(key=1).merge(days.assign(key=1), on="key").drop(columns="key")

# 6) 기상 LEFT 조인
base = base.merge(w, on=["지점","일시"], how="left")

# 7) 병해 LEFT 조인(해당 연도만, 병해 컬럼 자동 선별)
pcols = pest_cols(p_all)
p_y = p_all.loc[p_all["일시"].dt.year == y, ["지점ID","일시"] + pcols]
p_y = p_y.dropna(subset=pcols, how="all")  # 병해 전부 결측 행 제거(원치 않으면 주석)
out = base.merge(p_y, on=["지점ID","일시"], how="left")

# 8) 정리(불필요한 컬럼 제거는 선택)
drop_candidates = [c for c in ["지점명","위도","경도"] if c in out.columns]
out = out.drop(columns=drop_candidates).sort_values(["지점ID","일시"]).reset_index(drop=True)

# 9) 저장
save_path = os.path.join(OUT_DIR, f"joined_SAMPLE_{y}_{CROP}.csv")
out.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"[저장] {save_path}  rows={len(out):,}, cols={len(out.columns)}")


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/joined_samples/joined_SAMPLE_2018_APPLE.csv  rows=161,378, cols=11


### GDD10 2019 BEAN 샘플

In [36]:
# -*- coding: utf-8 -*-
"""
[1년 전용, 단일 관측지표 고정] 10℃ 기준 누적온량(GDD10) + 첫 발생 요약
"""

import os, re
import numpy as np
import pandas as pd

# ================== 설정 ==================
INPUT_FILE = "/Users/doyoung-gil/Desktop/연구실/데이터/joined_samples/joined_SAMPLE_2019_BEAN.csv"
OUT_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/derived"
BASE_TEMP  = 10.0  # 10℃ 기준
OBS_COL    = "탄저병-병든꼬투리수(협/300협)"
# =========================================

os.makedirs(OUT_DIR, exist_ok=True)

m = re.search(r"_(\d{4})_([A-Za-z]+)\.csv$", os.path.basename(INPUT_FILE))
YEAR_HINT = int(m.group(1)) if m else None
CROP      = m.group(2).upper() if m else "CROP"

def tmean_series(df: pd.DataFrame) -> pd.Series:
    if "평균기온(°C)" in df.columns:
        return pd.to_numeric(df["평균기온(°C)"], errors="coerce")
    tmin = pd.to_numeric(df.get("최저기온(°C)"), errors="coerce")
    tmax = pd.to_numeric(df.get("최고기온(°C)"), errors="coerce")
    return (tmin + tmax) / 2

# 1) 로드 & 연도 필터
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", parse_dates=["일시"])
if not {"지점ID","일시"}.issubset(df.columns):
    raise SystemExit("필수 컬럼('지점ID','일시')이 없습니다.")
if OBS_COL not in df.columns:
    raise SystemExit(f"'{OBS_COL}' 칼럼을 찾지 못했습니다. 파일의 실제 칼럼명을 확인하세요.")

df["연도"] = df["일시"].dt.year
YEAR = YEAR_HINT if YEAR_HINT is not None else int(df["연도"].mode().iloc[0])
df = df.loc[df["연도"] == YEAR].copy()
if df.empty:
    raise SystemExit(f"{YEAR}년에 해당하는 데이터가 없습니다.")

# 2) DD10 / GDD10 누적
tmean = tmean_series(df)
df["DD10"] = (tmean - BASE_TEMP).clip(lower=0)
df = df.sort_values(["지점ID","일시"])
df["GDD10_cum"] = df.groupby(["지점ID"], sort=False)["DD10"].cumsum()

# 🔹 소수 1자리로 반올림
df["DD10"] = df["DD10"].round(1)
df["GDD10_cum"] = df["GDD10_cum"].round(1)

# 3) 관측값/발생여부
df["관측값"]   = pd.to_numeric(df[OBS_COL], errors="coerce")
df["발생여부"] = (df["관측값"].fillna(0) > 0).astype(int)

# 4) 첫 발생 요약
has = df.loc[df["발생여부"]==1, ["지점ID","일시"]]
first = has.groupby("지점ID", as_index=False).agg(첫발생일=("일시","min"))
if not first.empty:
    gdd_at_first = (
        df.merge(first, on=["지점ID"], how="inner")
          .loc[lambda x: x["일시"].eq(x["첫발생일"]), ["지점ID","GDD10_cum"]]
          .rename(columns={"GDD10_cum":"첫발생_GDD10"})
    )
    first = first.merge(gdd_at_first, on="지점ID", how="left")
    first.insert(1, "연도", YEAR)
else:
    first = pd.DataFrame(columns=["지점ID","연도","첫발생일","첫발생_GDD10"])

# 5) 저장 (CSV 출력도 소수 1자리 고정)
daily_out = os.path.join(OUT_DIR, f"{CROP}_{YEAR}_GDD10_daily.csv")
first_out = os.path.join(OUT_DIR, f"{CROP}_{YEAR}_GDD10_first_event.csv")
df.to_csv(daily_out, index=False, encoding="utf-8-sig", float_format="%.1f")
first.to_csv(first_out, index=False, encoding="utf-8-sig", float_format="%.1f")

print("[저장]", daily_out,  "| rows=", len(df),    "| cols=", len(df.columns))
print("[저장]", first_out, "| rows=", len(first), "| cols=", len(first.columns))


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/derived/BEAN_2019_GDD10_daily.csv | rows= 158420 | cols= 16
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/derived/BEAN_2019_GDD10_first_event.csv | rows= 11 | cols= 4


In [39]:
print("HELLO1")

HELLO1
